<a href="https://colab.research.google.com/github/Ekrem-Guler/RAG-Based-Mathematical-Problem-Solver/blob/main/RAG_MATH.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import chromadb

In [2]:
from datasets import load_dataset

ds = load_dataset("EleutherAI/hendrycks_math", "counting_and_probability")
ds


DatasetDict({
    train: Dataset({
        features: ['problem', 'level', 'type', 'solution'],
        num_rows: 771
    })
    test: Dataset({
        features: ['problem', 'level', 'type', 'solution'],
        num_rows: 474
    })
})

In [37]:
ds_geo = load_dataset("EleutherAI/hendrycks_math", "geometry")
ds_geo

DatasetDict({
    train: Dataset({
        features: ['problem', 'level', 'type', 'solution'],
        num_rows: 870
    })
    test: Dataset({
        features: ['problem', 'level', 'type', 'solution'],
        num_rows: 479
    })
})

In [38]:
ds_alg = load_dataset("EleutherAI/hendrycks_math", "algebra")
ds_alg

algebra/train-00000-of-00001.parquet:   0%|          | 0.00/505k [00:00<?, ?B/s]

algebra/test-00000-of-00001.parquet:   0%|          | 0.00/353k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1744 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1187 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['problem', 'level', 'type', 'solution'],
        num_rows: 1744
    })
    test: Dataset({
        features: ['problem', 'level', 'type', 'solution'],
        num_rows: 1187
    })
})

In [15]:
sentences = [ds["train"][i]["solution"] for i in range(len(ds["train"]))]

In [33]:
sentences_test = [ds["test"][i]["solution"] for i in range(len(ds["test"]))]

In [41]:
sentences_geo = [ds_geo["train"][i]["solution"] for i in range(len(ds_geo["train"]))]

In [40]:
sentences_geo_test = [ds_geo["test"][i]["solution"] for i in range(len(ds_geo["test"]))]

In [42]:
sentences_alg = [ds_alg["train"][i]["solution"] for i in range(len(ds_alg["train"]))]

In [43]:
sentences_alg_test = [ds_alg["test"][i]["solution"] for i in range(len(ds_alg["test"]))]

In [16]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
embeddings = model.encode(sentences)
print(len(embeddings[0]))


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[-0.11805248 -0.23622514  0.17850831  0.02073083 -0.16442865 -0.0482368
  0.4132088   0.30421495  0.18911302  0.24830246  0.3812828  -0.39283186
  0.0079609  -0.02743122 -0.15021105 -0.03145849 -0.19074869  0.19870983
 -0.40783298 -0.17560978  0.4517256  -0.33977872 -0.26582283  0.00127624
  0.42748147  0.20481092 -0.2855409  -0.01592438  0.0107084   0.04854083
  0.27183813  0.29464808  0.3281282  -0.09084985 -0.17075576  0.08693807
 -0.18125415 -0.06313168 -0.147051    0.33946937 -0.23665074  0.38349703
  0.1764757  -0.11638767  0.10169271 -0.35124874 -0.10380503 -0.14852099
 -0.04177853 -0.2066248  -0.15378337  0.15319298 -0.08954052 -0.06033001
 -0.14563851 -0.18134724 -0.14220254 -0.05683684  0.24154875 -0.20378685
  0.21695364  0.01328294 -0.03973462  0.08955727  0.18781216 -0.21025099
  0.1339511  -0.16075318  0.01616288  0.41389734 -0.13653094  0.08357796
 -0.1510032  -0.39219692  0.2584608   0.1893684  -0.03836063 -0.08381848
  0.30202374  0.1851726  -0.0230588   0.04571532  0.

In [44]:
model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
embeddings_geo = model.encode(sentences_geo)
print(len(embeddings_geo[0]))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


384


In [73]:
model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
embeddings_alg = model.encode(sentences_alg)
print(len(embeddings_alg[0]))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


384


In [67]:
"""
chroma_client = chromadb.Client()
collection_math = chroma_client.create_collection(name="math_collection",configuration={"hnsw": {"space": "cosine"}})
"""

In [68]:
"""
collection_math.add(
    documents=sentences,
    ids = [str(i) for i in range(771)],
    embeddings=embeddings.tolist()
)
"""

In [70]:
"""
collection_math.add(
    documents=sentences_geo,
    ids = [str(i+771) for i in range(870)],
    embeddings=embeddings_geo.tolist()
)
"""

In [74]:
"""
collection_math.add(
    documents=sentences_alg,
    ids = [str(i+771+870) for i in range(1744)],
    embeddings=embeddings_alg.tolist()
)
"""

In [75]:
results = collection_math.query(
    query_texts=[
        "How many natural numbers greater than 6 but less than 60 are relatively prime to 15?"
    ],
    n_results=10,
    include=["embeddings", "documents", "distances"]
)

retrieved_solution = results['documents'][0][0]
print(f"Retrieved Solution: {retrieved_solution}")
for i in range(len(results['ids'][0])):
    print(f"ID: {results['ids'][0][i]}")
    print(f"Distance: {results['distances'][0][i]}") # Lower is better!
    print(f"Content snippet: {results['documents'][0][i][:50]}...")

Retrieved Solution: It might be easier to find the integers less than or equal to 30 which are NOT relatively prime to 30. They include 2, 4, 6, 8, 10, $\ldots$, 28, 30, or 15 even integers. They also include 3, 9, 15, 21, 27, or the odd multiples of 3. And also, 5, 25, the multiples of 5 relatively prime to 2 and 3. So we have a total of $15+5+2 = 22$ numbers sharing a factor with 30. So there are 8 relatively prime integers, giving us a ratio of $\frac{8}{30} = \boxed{\frac{4}{15}}$.

Notice that the prime divisors of 30 are 2, 3, and 5, and we have $$30\left(1-\frac{1}{2}\right)\left(1-\frac{1}{3}\right)\left(1-\frac{1}{5}\right) = 30 \cdot \frac{1}{2} \cdot \frac{2}{3} \cdot \frac{4}{5} = 8,$$ which equals the number of positive integers less than 30 that are relatively prime to 30.  Is this a coincidence?
ID: 547
Distance: 0.7029851675033569
Content snippet: It might be easier to find the integers less than ...
ID: 559
Distance: 0.724702000617981
Content snippet: The first 10 prim